# Silver to Gold (cópia do Gabriel)

Construção das 12 tabelas analíticas da camada **Gold** (`v_energy.silver` → `v_energy.gold_gabriel`) seguindo estritamente o contrato [`contracts/gold_schema.yaml`](../../contracts/gold_schema.yaml) (**v0.3.0**) e o dicionário [`docs/dicionario-de-dados.md`](../../docs/dicionario-de-dados.md). Pela `DEC-011`, Alisson, Arthur e Gabriel mantêm cada um sua cópia independente para comparação antes de consolidar a versão oficial em `v_energy.gold`.

**Como executar**
- `Run All` processa todas as tabelas Gold em ordem de dependência e executa a reconciliação cruzada no final.
- Com `gravar = nao` (padrão), todas as tabelas são construídas em memória e **validadas contra os checks e travas do contrato**, sem gravar no catálogo.
- Com `gravar = sim`, cada tabela validada é gravada em `v_energy.<schema_gold>.<tabela>` (Delta, `overwrite`) e conferida em seguida.
- O widget `schema_gold` (padrão `gold_gabriel`) permite gravar na cópia individual (`DEC-011`) ou em `gold` (versão final / workspace isolado).
- Com `exportar_csv = sim`, exporta os 12 arquivos CSV do contrato (`UTF-8`, vírgula, nulo vazio `""`, booleanos `true`/`false`, arrays separados por `;` — `DEC-004`) para `csv_export_path`.

**Decisões de Arquitetura e Negócio Implementadas**
- `DEC-004` / `DEC-010` / `DEC-011`: `competencia` em texto `AAAA-MM`, dinheiro em `decimal(14,2)`, energia em `decimal(16,3)`, comparação de 3 cópias independentes de Dados.
- `DEC-014`: espelhamento estrito de nomes, tipos, nulidade, domínios e chaves de `contracts/gold_schema.yaml` v0.3.0.
- `DEC-022`: mascaramento de dados pessoais (`titular_mascarado` com iniciais sem partículas e `documento_mascarado` com 3 últimos dígitos; dados brutos de LGPD não chegam à Gold).
- `DEC-023`: regra validada de `kwh_faturado` (`leitura`, `leitura_atual_troca_medidor`, `consumo_absoluto`), três medidas de receita (`receita_faturada`, `receita_energia`, `tributos`) e ponte de reconciliação (`gold_reconciliacao_faturamento`).
- `DEC-027`: contrato v0.3.0 com as tabelas da UC 360 e Perdas (`gold_medidor_uc`, `gold_interrupcoes_uc`, `gold_atendimentos_uc`, `gold_inspecoes_uc`) e ampliação da chave de `gold_kpis_segmento_mensal` com `cod_alimentador` e `tipo_medidor`.
- `DEC-028` / `DEC-031`: decomposição PVM em lote gravando todos os pares de mesmos meses em anos diferentes de 1 a 12 meses (permitindo a seleção dinâmica na rota `/receita/decomposicao/periodos`).
- `DEC-029`: método exato do efeito bandeira e reajuste da tarifa base (bandeira como adicional fixo por kWh).
- `DEC-030`: regras regulatórias e de qualidade:
  - `n_uc`: UCs **faturadas** na competência (denominador do DEC/FEC, PRODIST Módulo 8);
  - `OCO-04`: ocorrências antes de `data_ligacao` da UC excluídas da Gold;
  - `GD-02`: potência em Wp convertida (/1000) apenas quando confirmada pelas faixas das UCs normais (senão vazia);
  - Vazio × zero nos recortes: `receita_arrecadada` nula sem faturas, zero quando não paga; `soma_notas` nula quando sem avaliações (`n_notas = 0`);
  - Fila de suspeitas calibrada: UCs com irregularidade confirmada anterior e UCs inspecionadas sem irregularidade nos últimos 12 meses ficam fora da fila.

**Confidencial (`DEC-024`):** limpe os outputs das células antes de exportar ou commitar o notebook.

## 0. Configuração e Funções de Validação
Execute as duas células desta seção antes de rodar qualquer tabela.

In [ ]:
import traceback
from datetime import datetime

from pyspark.sql import Window
from pyspark.sql import functions as F

dbutils.widgets.dropdown("gravar", "nao", ["nao", "sim"], "1. Gravar na Gold?")
dbutils.widgets.text("schema_gold", "gold_gabriel", "2. Schema Gold destino")  # cópia do Gabriel (DEC-011); v_energy.gold fica para a versão final
dbutils.widgets.dropdown("exportar_csv", "nao", ["nao", "sim"], "3. Exportar CSVs?")
dbutils.widgets.text("csv_export_path", "/Volumes/v_energy/default/gold_csv", "4. Pasta exportação CSV")

GRAVAR = dbutils.widgets.get("gravar") == "sim"
SCHEMA_GOLD = dbutils.widgets.get("schema_gold").strip() or "gold_gabriel"
EXPORTAR_CSV = dbutils.widgets.get("exportar_csv") == "sim"
CSV_EXPORT_PATH = dbutils.widgets.get("csv_export_path").strip().rstrip("/")

CATALOG = "v_energy"
SILVER = f"{CATALOG}.silver"
GOLD = f"{CATALOG}.{SCHEMA_GOLD}"

# 12 tabelas oficiais do Contrato da Gold v0.3.0 (DEC-027)
TABELAS_CONTRATO_BACKEND = [
    "gold_dim_uc",
    "gold_fato_faturamento_mensal",
    "gold_reconciliacao_faturamento",
    "gold_continuidade_alimentador_mensal",
    "gold_kpis_segmento_mensal",
    "gold_kpis_mensais",
    "gold_receita_decomposicao",
    "gold_fila_suspeitas",
]
TABELAS_DETALHE_UC = [
    "gold_medidor_uc",
    "gold_interrupcoes_uc",
    "gold_atendimentos_uc",
    "gold_inspecoes_uc",
]
TABELAS_ESPERADAS = TABELAS_CONTRATO_BACKEND + TABELAS_DETALHE_UC

if GRAVAR:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD}")

RESULTADOS = {}
GOLD_DF = {}
CACHE_AUX = {}
print(f"Silver: {SILVER} | Gold: {GOLD} | gravar = {'sim' if GRAVAR else 'nao'} | exportar_csv = {'sim' if EXPORTAR_CSV else 'nao'}")

In [ ]:
# ---------- Funções auxiliares de tipos, travas e execução ----------

ACRESCIMO_BANDEIRA = {
    "verde": 0.0,
    "amarela": 0.01885,
    "vermelha_p1": 0.04463,
    "vermelha_p2": 0.07877,
}

PARTICULAS_NOME = ["DA", "DE", "DO", "DAS", "DOS", "E"]


def money(col_or_expr):
    """Converte expressão numérica para DECIMAL(14,2) (tipo money do contrato, DEC-010)."""
    c = F.col(col_or_expr) if isinstance(col_or_expr, str) else col_or_expr
    return F.round(c, 2).cast("decimal(14,2)")


def energy(col_or_expr):
    """Converte expressão numérica para DECIMAL(16,3) (tipo energy_kwh do contrato, DEC-010)."""
    c = F.col(col_or_expr) if isinstance(col_or_expr, str) else col_or_expr
    return F.round(c, 3).cast("decimal(16,3)")


def ratio(col_or_expr):
    """Converte expressão numérica para DOUBLE (tipo ratio do contrato)."""
    c = F.col(col_or_expr) if isinstance(col_or_expr, str) else col_or_expr
    return c.cast("double")


def exigir_zero(df, regras):
    """regras = {nome: condição}. Toda condição de erro precisa ter 0 linhas."""
    r = df.select(*[
        F.sum(F.coalesce(cond, F.lit(False)).cast("int")).alias(nome)
        for nome, cond in regras.items()
    ]).first().asDict()
    violadas = {nome: n for nome, n in r.items() if n}
    assert not violadas, f"regras violadas (linhas): {violadas}"


def exigir_sem_nulos(df, colunas_obrigatorias):
    """Garante que nenhuma coluna nullable=false no contrato tenha NULL."""
    regras = {f"nulo_em_{c}": F.col(c).isNull() for c in colunas_obrigatorias}
    exigir_zero(df, regras)


def exigir_chave_unica(df, chave):
    exigir_sem_nulos(df, chave)
    total = df.count()
    unicos = df.select(*chave).distinct().count()
    assert total == unicos, f"chave primária {chave} duplicada: {total - unicos:,} linhas"
    return total


def exigir_colunas_exatas(df, esperadas):
    """Garante que o DataFrame tem exatamente as colunas do contrato, na mesma ordem."""
    atuais = list(df.columns)
    assert atuais == list(esperadas), (
        f"Colunas divergentes do contrato!\n"
        f"Esperadas: {list(esperadas)}\n"
        f"Atuais:    {atuais}"
    )


def exigir_fk_uc(df, nome, col_uc="cod_uc"):
    """Chave estrangeira lógica para gold_dim_uc (conventions.foreign_keys: logical)."""
    dim_uc = obter_gold("gold_dim_uc").select(F.col("cod_uc").alias(col_uc))
    orfaos = df.select(col_uc).distinct().join(dim_uc, col_uc, "left_anti").count()
    assert orfaos == 0, f"{nome}: {orfaos:,} UCs sem correspondência em gold_dim_uc"


def obter_silver(nome):
    return spark.table(f"{SILVER}.{nome}")


def obter_gold(nome):
    return GOLD_DF[nome] if nome in GOLD_DF else spark.table(f"{GOLD}.{nome}")


def executar_gold(nome, construir):
    """Constrói e valida a tabela Gold; só grava se gravar = sim e passar em todas as travas."""
    print(f"=== {nome}")
    try:
        df, info = construir()
        GOLD_DF[nome] = df
        if GRAVAR:
            destino = f"{GOLD}.{nome}"
            df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(destino)
            gravadas = spark.table(destino).count()
            assert gravadas == info["linhas"], f"gravadas {gravadas:,} != validadas {info['linhas']:,}"
            GOLD_DF[nome] = spark.table(destino)
            status = "gravada e conferida"
        else:
            status = "validada (sem gravar)"
        RESULTADOS[nome] = {"status": status, **info}
        print(f"[OK] {nome}: {status} | {info}")
    except Exception as e:
        RESULTADOS[nome] = {"status": "FALHA", "erro": f"{type(e).__name__}: {e}"}
        print(f"[FALHA] {nome}: nada foi gravado.")
        traceback.print_exc()


def obter_leituras_tratadas():
    """Aplica a regra validada de kwh_faturado, origem_kwh, tipo_medidor e dias_faturados (DEC-023, DEC-027)."""
    if "leituras_tratadas" in CACHE_AUX:
        return CACHE_AUX["leituras_tratadas"]

    leituras = (
        obter_silver("leituras")
        .select(
            F.col("id_uc"),
            F.col("mes_referencia"),
            F.col("id_medidor"),
            F.col("data_leitura"),
            F.lower(F.col("tipo_leitura")).alias("tipo_leitura"),
            F.col("leitura_anterior"),
            F.col("leitura_atual"),
            F.col("consumo_kwh"),
        )
    )
    medidores = obter_silver("medidores").select(
        F.col("id_medidor"),
        F.lower(F.col("tipo")).alias("tipo_medidor"),
    )

    w_uc = Window.partitionBy("id_uc").orderBy("mes_referencia")
    leituras_enriquecidas = (
        leituras
        .join(medidores, "id_medidor", "left")
        .withColumn("dias_faturados", F.datediff(F.col("data_leitura"), F.lag("data_leitura").over(w_uc)))
    )

    trocas_competencia = (
        obter_silver("medidores")
        .select(
            F.col("id_uc"),
            F.date_format(F.col("data_instalacao"), "yyyy-MM").alias("mes_referencia"),
        )
        .dropna()
        .distinct()
        .withColumn("_teve_instalacao_mes", F.lit(True))
    )

    df = (
        leituras_enriquecidas
        .join(trocas_competencia, ["id_uc", "mes_referencia"], "left")
        .withColumn("_teve_instalacao_mes", F.coalesce(F.col("_teve_instalacao_mes"), F.lit(False)))
        .withColumn(
            "origem_kwh",
            F.when(F.col("consumo_kwh") >= 0, F.lit("leitura"))
             .when(F.col("_teve_instalacao_mes"), F.lit("leitura_atual_troca_medidor"))
             .otherwise(F.lit("consumo_absoluto")),
        )
        .withColumn(
            "kwh_faturado",
            energy(
                F.when(F.col("origem_kwh") == "leitura", F.col("consumo_kwh"))
                 .when(F.col("origem_kwh") == "leitura_atual_troca_medidor", F.col("leitura_atual"))
                 .otherwise(F.abs(F.col("consumo_kwh")))
            ),
        )
        .withColumn("kwh_leitura", energy(F.col("consumo_kwh")))
        .drop("_teve_instalacao_mes")
    )
    CACHE_AUX["leituras_tratadas"] = df
    return df


def obter_base_uc_mes():
    """Constrói o grão UC × competência com todos os componentes somáveis (DEC-027, DEC-030).

    Base unificada que alimenta gold_continuidade_alimentador_mensal, gold_kpis_segmento_mensal
    e gold_kpis_mensais, garantindo reconciliação matemática exata por construção.
    """
    if "base_uc_mes" in CACHE_AUX:
        return CACHE_AUX["base_uc_mes"]

    fato = obter_gold("gold_fato_faturamento_mensal")
    comp = fato.select("competencia").distinct()

    u = obter_silver("unidades_consumidoras")
    exigir_zero(u, {"uc_sem_data_ligacao": F.col("data_ligacao").isNull()})

    gd = obter_silver("geracao_distribuida").select(
        F.col("id_uc").alias("cod_uc"),
        F.date_format(F.col("data_conexao"), "yyyy-MM").alias("_mes_conexao"),
    )

    dim = obter_gold("gold_dim_uc").select(
        "cod_uc", "municipio", "bairro", "classe_cadastro", "classe", "cod_alimentador", "data_ligacao"
    )

    ucs = (
        dim
        .withColumn("_mes_ligacao", F.date_format("data_ligacao", "yyyy-MM"))
        .join(gd, "cod_uc", "left")
    )

    base = (
        comp
        .crossJoin(ucs)
        .filter(F.col("_mes_ligacao") <= F.col("competencia"))
        .withColumn("possui_gd", F.coalesce(F.col("_mes_conexao") <= F.col("competencia"), F.lit(False)))
        .drop("_mes_ligacao", "_mes_conexao", "data_ligacao")
    )

    # Medidor vigente na competência (DEC-027)
    medidores = obter_silver("medidores").select(
        F.col("id_uc").alias("cod_uc"),
        F.lower(F.col("tipo")).alias("_tipo"),
        F.date_format(F.col("data_instalacao"), "yyyy-MM").alias("_mes_instalacao"),
        F.date_format(F.col("data_retirada"), "yyyy-MM").alias("_mes_retirada"),
    )
    vigente = (
        base
        .select("cod_uc", "competencia")
        .join(medidores, "cod_uc")
        .filter(
            (F.col("_mes_instalacao") <= F.col("competencia"))
            & (F.col("_mes_retirada").isNull() | (F.col("_mes_retirada") > F.col("competencia")))
        )
        .groupBy("cod_uc", "competencia")
        .agg(F.count(F.lit(1)).alias("_vigentes"), F.first("_tipo").alias("tipo_medidor"))
    )
    exigir_zero(vigente, {"mais_de_um_medidor_vigente": F.col("_vigentes") > 1})

    base = (
        base
        .join(vigente.drop("_vigentes"), ["cod_uc", "competencia"], "left")
        .fillna("sem_medidor", ["tipo_medidor"])
    )

    # Ocorrências de rede: OCO-04 exclui ocorrências antes da ligação (DEC-030: Gold + excluídas = Silver)
    oc_silver = (
        obter_silver("ocorrencias_rede")
        .select(
            F.col("id_uc").alias("cod_uc"),
            F.date_format(F.col("data_abertura"), "yyyy-MM").alias("competencia"),
            (F.col("duracao_min") / 60.0).alias("duracao_horas"),
        )
        .join(dim.select("cod_uc", F.date_format("data_ligacao", "yyyy-MM").alias("_mes_lig")), "cod_uc")
    )
    CACHE_AUX["ocorrencias_antes_ligacao"] = oc_silver.filter(F.col("competencia") < F.col("_mes_lig")).count()

    oc = (
        oc_silver
        .filter(F.col("competencia") >= F.col("_mes_lig"))
        .drop("_mes_lig")
        .groupBy("cod_uc", "competencia")
        .agg(
            ratio(F.sum("duracao_horas")).alias("horas_interrompidas"),
            F.count(F.lit(1)).cast("int").alias("n_interrupcoes"),
        )
    )

    # Chamados de atendimento (soma_notas preservada em Decimal na base antes da agregação final)
    chamados = (
        obter_silver("atendimentos")
        .groupBy(F.col("id_uc").alias("cod_uc"), F.date_format(F.col("data_abertura"), "yyyy-MM").alias("competencia"))
        .agg(
            F.count(F.lit(1)).cast("int").alias("n_chamados"),
            F.sum("nota_satisfacao").alias("soma_notas"),
            F.count("nota_satisfacao").cast("int").alias("n_notas"),
        )
    )

    nao_paga = F.col("status_pagamento").isin("em_aberto", "em_atraso")
    faturas = fato.select(
        "cod_uc",
        "competencia",
        "kwh_faturado",
        "receita_faturada",
        "receita_energia",
        "tributos",
        "receita_arrecadada",
        "receita_bandeira",
        F.lit(1).alias("n_faturas"),
        nao_paga.cast("int").alias("n_faturas_nao_pagas"),
        F.when(nao_paga, F.col("receita_faturada")).otherwise(money(F.lit(0))).alias("valor_nao_pago"),
        (F.col("status_pagamento") == "paga_com_atraso").cast("int").alias("n_faturas_pagas_com_atraso"),
        F.coalesce("dias_em_atraso", F.lit(0)).alias("soma_dias_em_atraso"),
        F.lit(1).alias("n_leituras"),
        (F.col("tipo_leitura") == "estimada").cast("int").alias("n_leituras_estimadas"),
    )

    componentes_zerar = [
        "kwh_faturado", "receita_faturada", "receita_energia", "tributos", "receita_arrecadada",
        "horas_interrompidas", "n_interrupcoes", "n_chamados", "soma_notas", "n_notas", "receita_bandeira",
        "n_faturas", "n_faturas_nao_pagas", "valor_nao_pago", "n_faturas_pagas_com_atraso", "soma_dias_em_atraso",
        "n_leituras", "n_leituras_estimadas"
    ]

    base = (
        base
        .join(faturas, ["cod_uc", "competencia"], "left")
        .join(oc, ["cod_uc", "competencia"], "left")
        .join(chamados, ["cod_uc", "competencia"], "left")
        .fillna(0, componentes_zerar)
        .filter((F.col("n_faturas") > 0) | (F.col("n_interrupcoes") > 0) | (F.col("n_chamados") > 0))
    )

    CACHE_AUX["base_uc_mes"] = base
    return base

## 1. `gold_dim_uc`
Dimensão de unidades consumidoras (`1 linha por UC`, chave `cod_uc`).

**Regras aplicadas (`DEC-022`, `DEC-023`, `DEC-027`, `DEC-030`):**
- `cod_uc` ← `id_uc`; `classe` ← `subgrupo_tarifario` (`B1`, `B2`, `B3`, `A4`); `classe_cadastro` ← `classe` (`Residencial`, `Comercial`, `Industrial`, `Rural`, `Poder Público`); `cod_alimentador` ← `alimentador`.
- `possui_gd`: verdadeiro quando a UC existe em `silver.geracao_distribuida`.
- `potencia_kwp`: potência em kWp para UCs com GD (`NULL` sem GD). **Regra DEC-030 (GD-02):** registros com `flag_potencia_anomala` (em Wp) são convertidos para kWp (`/ 1000.0`) **quando o dado confirma** (valor convertido e energia injetada por kWp dentro da faixa das UCs normais); se não confirmam, ficam `NULL`.
- `data_ligacao` (not null): data em que a UC foi ligada. A UC passa a contar no `n_uc` a partir da primeira competência faturada (`DEC-030`).
- `data_conexao_gd`, `modalidade_gd` (`geracao_propria_uc`, `autoconsumo_remoto`, `geracao_compartilhada`) e `energia_injetada_kwh_mes`: atributos da geração própria (vazios sem GD).
- `consumo_medio_kwh_12m`: média mensal de `kwh_faturado` nos últimos 12 meses da base. Fica `NULL` para UCs sem leituras nessa janela (`DEC-023`).
- `titular_mascarado`: apenas as iniciais do titular sem partículas (`DA`, `DE`, `DO`, `DAS`, `DOS`, `E`), ex.: `"M.S."` (`DEC-022`).
- `documento_mascarado`: documento mascarado mantendo apenas os 3 últimos dígitos (`***.***.**X-XX`). Nulo para UCs com `flag_documento_repetido` (`UC-12`).

In [ ]:
COLUNAS_DIM_UC = [
    "cod_uc",
    "municipio",
    "bairro",
    "classe",
    "classe_cadastro",
    "cod_alimentador",
    "possui_gd",
    "consumo_medio_kwh_12m",
    "titular_mascarado",
    "documento_mascarado",
    "potencia_kwp",
    "data_ligacao",
    "data_conexao_gd",
    "modalidade_gd",
    "energia_injetada_kwh_mes",
]

MODALIDADE_GD_MAP = {
    "GERACAO NA PROPRIA UC": "geracao_propria_uc",
    "AUTOCONSUMO REMOTO": "autoconsumo_remoto",
    "GERACAO COMPARTILHADA": "geracao_compartilhada",
}


def construir_gold_dim_uc():
    ucs = obter_silver("unidades_consumidoras")
    gd = obter_silver("geracao_distribuida")
    leituras = obter_leituras_tratadas()

    max_row = leituras.agg(F.max("mes_referencia")).first()
    ultima_comp = max_row[0] if max_row and max_row[0] else None
    if ultima_comp:
        inicio_12m = F.date_format(F.add_months(F.to_date(F.lit(f"{ultima_comp}-01")), -11), "yyyy-MM")
        consumo_12m = (
            leituras
            .filter((F.col("mes_referencia") >= inicio_12m) & (F.col("mes_referencia") <= ultima_comp))
            .groupBy("id_uc")
            .agg(energy(F.avg("kwh_faturado")).alias("consumo_medio_kwh_12m"))
        )
    else:
        inicio_12m = None
        consumo_12m = (
            leituras
            .groupBy("id_uc")
            .agg(energy(F.avg("kwh_faturado")).alias("consumo_medio_kwh_12m"))
        )

    # GD-02 (DEC-030): faixa das UCs normais para confirmação estatística de Wp -> kWp
    gd_com_inj = gd.withColumn("_inj_por_kwp", F.col("energia_injetada_kwh_media_mensal") / F.col("potencia_kwp"))
    normal_stats = (
        gd_com_inj
        .filter(~F.col("flag_potencia_anomala"))
        .agg(
            F.min("potencia_kwp").alias("pot_min"),
            F.max("potencia_kwp").alias("pot_max"),
            F.min("_inj_por_kwp").alias("inj_min"),
            F.max("_inj_por_kwp").alias("inj_max"),
        )
        .first()
    )

    pot_conv = F.col("potencia_kwp") / 1000
    inj_conv = F.col("energia_injetada_kwh_media_mensal") / pot_conv
    wp_confirmado = (
        pot_conv.between(normal_stats["pot_min"], normal_stats["pot_max"])
        & inj_conv.between(normal_stats["inj_min"], normal_stats["inj_max"])
    )

    mapa_modalidade_expr = F.create_map(*[F.lit(x) for kv in MODALIDADE_GD_MAP.items() for x in kv])

    gd_tratada = (
        gd
        .withColumn("_possui_gd", F.lit(True))
        .withColumn("_wp_convertida", F.col("flag_potencia_anomala") & wp_confirmado)
        .withColumn(
            "potencia_kwp",
            F.when(~F.col("flag_potencia_anomala"), ratio(F.col("potencia_kwp")))
             .when(F.col("_wp_convertida"), ratio(pot_conv))
             .otherwise(F.lit(None).cast("double")),
        )
        .withColumn("modalidade_gd", mapa_modalidade_expr[F.col("modalidade")])
        .withColumn("data_conexao_gd", F.col("data_conexao").cast("date"))
        .withColumn("energia_injetada_kwh_mes", energy(F.col("energia_injetada_kwh_media_mensal")))
        .select(
            F.col("id_uc"),
            "_possui_gd",
            "_wp_convertida",
            "potencia_kwp",
            "data_conexao_gd",
            "modalidade_gd",
            "energia_injetada_kwh_mes",
        )
    )
    n_wp_convertidas = gd_tratada.filter(F.col("_wp_convertida")).count()
    gd_tratada = gd_tratada.drop("_wp_convertida")

    # Mascaramento de titular (iniciais sem partículas: 'JOAO DA SILVA' -> 'J.S.')
    palavras = F.filter(F.split(F.trim(F.col("nome_titular")), r"\s+"), lambda p: ~p.isin(*PARTICULAS_NOME))
    iniciais_expr = F.array_join(F.transform(palavras, lambda p: F.concat(F.substring(p, 1, 1), F.lit("."))), "")

    # Documento mascarado (apenas 3 últimos dígitos expostos, nulo na UC-12)
    doc_limpo = F.trim(F.col("cpf_cnpj"))
    doc_mascarado_expr = (
        F.when(F.col("flag_documento_repetido") | doc_limpo.isNull() | (F.length(doc_limpo) < 3), F.lit(None).cast("string"))
         .otherwise(F.expr("concat(repeat('*', length(cpf_cnpj) - 3), right(cpf_cnpj, 3))"))
    )

    df = (
        ucs
        .join(gd_tratada, "id_uc", "left")
        .join(consumo_12m, "id_uc", "left")
        .select(
            F.col("id_uc").alias("cod_uc"),
            F.col("municipio"),
            F.col("bairro"),
            F.col("subgrupo_tarifario").alias("classe"),
            F.col("classe").alias("classe_cadastro"),
            F.col("alimentador").alias("cod_alimentador"),
            F.coalesce(F.col("_possui_gd"), F.lit(False)).alias("possui_gd"),
            F.col("consumo_medio_kwh_12m"),
            F.when(F.size(palavras) > 0, iniciais_expr).alias("titular_mascarado"),
            doc_mascarado_expr.alias("documento_mascarado"),
            F.col("potencia_kwp"),
            F.col("data_ligacao").cast("date").alias("data_ligacao"),
            F.col("data_conexao_gd"),
            F.col("modalidade_gd"),
            F.col("energia_injetada_kwh_mes"),
        )
        .select(*COLUNAS_DIM_UC)
        .orderBy("cod_uc")
    )

    exigir_colunas_exatas(df, COLUNAS_DIM_UC)
    total = exigir_chave_unica(df, ["cod_uc"])
    exigir_sem_nulos(df, ["cod_uc", "municipio", "bairro", "classe", "classe_cadastro", "cod_alimentador", "possui_gd", "data_ligacao"])

    exigir_zero(df, {
        "gd_sem_data_conexao": F.col("possui_gd") & F.col("data_conexao_gd").isNull(),
        "gd_sem_modalidade": F.col("possui_gd") & F.col("modalidade_gd").isNull(),
        "nao_gd_com_potencia": (~F.col("possui_gd")) & F.col("potencia_kwp").isNotNull(),
        "nao_gd_com_conexao": (~F.col("possui_gd")) & F.col("data_conexao_gd").isNotNull(),
        "consumo_negativo": F.col("consumo_medio_kwh_12m") < 0,
        "documento_com_mais_de_3_digitos": F.length(F.regexp_replace("documento_mascarado", r"[^0-9]", "")) > 3,
    })

    return df, {
        "linhas": total,
        "com_gd": df.filter("possui_gd").count(),
        "sem_consumo_medio": df.filter(F.col("consumo_medio_kwh_12m").isNull()).count(),
        "potencia_convertida_de_wp": n_wp_convertidas,
        "gd_sem_potencia": df.filter(F.col("possui_gd") & F.col("potencia_kwp").isNull()).count(),
        "janela_consumo_medio": f"12 competências até {ultima_comp}" if ultima_comp else "sem leituras",
    }


executar_gold("gold_dim_uc", construir_gold_dim_uc)


## 2. `gold_fato_faturamento_mensal`
Faturamento por UC e competência (`grão: cod_uc × competencia`).

**Regras aplicadas (`DEC-023`, `DEC-027`, `DEC-029`):**
- Pareamento estrito 1:1 entre `silver.faturas` (apenas UCs do cadastro: `~flag_uc_orfa`) e `silver.leituras`. As faturas órfãs vão para `gold_reconciliacao_faturamento`.
- `kwh_faturado`, `kwh_leitura`, `origem_kwh`, `tipo_leitura`, `tipo_medidor` e `dias_faturados` calculados pela regra validada.
- Medidas de receita: `receita_faturada`, `receita_energia`, `tributos` e `receita_arrecadada` (`0.00` quando não paga).
- `receita_bandeira` (`DEC-029`): acréscimo regulatório da bandeira tarifária (`kwh_faturado × adicional`).
- `status_pagamento` e `dias_em_atraso` calculados contra a `data_corte`.

In [ ]:
COLUNAS_FATO_FATURAMENTO = [
    "cod_uc",
    "competencia",
    "kwh_faturado",
    "kwh_leitura",
    "origem_kwh",
    "receita_faturada",
    "receita_energia",
    "tributos",
    "receita_arrecadada",
    "tarifa_media",
    "bandeira",
    "data_vencimento",
    "data_pagamento",
    "status_pagamento",
    "dias_em_atraso",
    "receita_bandeira",
    "tipo_leitura",
    "tipo_medidor",
    "dias_faturados",
]

MAPA_BANDEIRA = {
    "Verde": "verde",
    "Amarela": "amarela",
    "Vermelha P1": "vermelha_p1",
    "Vermelha P2": "vermelha_p2",
}


def construir_gold_fato_faturamento_mensal():
    faturas_todas = obter_silver("faturas")
    leituras = obter_leituras_tratadas()

    data_corte = faturas_todas.select(
        F.greatest(F.max("data_emissao"), F.max("data_pagamento")).alias("corte")
    ).first()["corte"]
    CACHE_AUX["data_corte"] = data_corte

    faturas_validas = (
        faturas_todas
        .filter(~F.col("flag_uc_orfa"))
        .withColumnRenamed("id_uc", "cod_uc")
        .withColumnRenamed("mes_referencia", "competencia")
    )
    mapa_bandeira_expr = F.create_map(*[F.lit(x) for kv in MAPA_BANDEIRA.items() for x in kv])
    mapa_acrescimo_expr = F.create_map(*[F.lit(x) for kv in ACRESCIMO_BANDEIRA.items() for x in kv])

    df_join = faturas_validas.join(
        leituras.select(
            F.col("id_uc").alias("cod_uc"),
            F.col("mes_referencia").alias("competencia"),
            "kwh_faturado",
            "kwh_leitura",
            "origem_kwh",
            "tipo_leitura",
            "tipo_medidor",
            "dias_faturados",
        ),
        ["cod_uc", "competencia"],
        "inner",
    )

    venc, pag, corte_col = F.col("data_vencimento"), F.col("data_pagamento"), F.lit(data_corte)
    status_pag_expr = (
        F.when(pag.isNotNull() & (pag <= venc), F.lit("paga"))
         .when(pag.isNotNull(), F.lit("paga_com_atraso"))
         .when(venc >= corte_col, F.lit("em_aberto"))
         .otherwise(F.lit("em_atraso"))
    )

    dias_atraso_expr = (
        F.when(status_pag_expr == "paga_com_atraso", F.datediff(pag, venc))
         .when(status_pag_expr == "em_atraso", F.datediff(corte_col, venc))
         .otherwise(F.lit(None).cast("int"))
    )

    bandeira_norm = mapa_bandeira_expr[F.col("bandeira_tarifaria")]
    acrescimo_kwh = F.coalesce(mapa_acrescimo_expr[bandeira_norm], F.lit(0.0))
    receita_bandeira_expr = money(F.round(F.col("kwh_faturado") * acrescimo_kwh, 2))

    df = (
        df_join
        .withColumn("bandeira", bandeira_norm)
        .withColumn("status_pagamento", status_pag_expr)
        .withColumn("dias_em_atraso", dias_atraso_expr)
        .withColumn("receita_faturada", money("valor_total"))
        .withColumn("receita_energia", money("valor_energia"))
        .withColumn("tributos", money("tributos"))
        .withColumn("receita_bandeira", receita_bandeira_expr)
        .withColumn(
            "receita_arrecadada",
            F.when(pag.isNotNull(), money("valor_total")).otherwise(money(F.lit(0.0))),
        )
        .withColumn(
            "tarifa_media",
            ratio(
                F.when(F.col("kwh_faturado") > 0, F.col("receita_energia") / F.col("kwh_faturado"))
                 .otherwise(F.lit(None).cast("double"))
            ),
        )
        .select(*COLUNAS_FATO_FATURAMENTO)
        .orderBy("cod_uc", "competencia")
    )

    exigir_colunas_exatas(df, COLUNAS_FATO_FATURAMENTO)
    total = exigir_chave_unica(df, ["cod_uc", "competencia"])
    exigir_sem_nulos(df, [
        "cod_uc", "competencia", "kwh_faturado", "kwh_leitura", "origem_kwh",
        "receita_faturada", "receita_energia", "tributos", "data_vencimento",
        "status_pagamento", "receita_bandeira", "tipo_leitura", "tipo_medidor"
    ])
    exigir_fk_uc(df, "gold_fato_faturamento_mensal")

    exigir_zero(df, {
        "fechamento_faturada_energia_tributos": F.abs(F.col("receita_faturada") - F.col("receita_energia") - F.col("tributos")) > 0.01,
        "bandeira_verde_com_receita_bandeira": (F.col("bandeira") == "verde") & (F.col("receita_bandeira") != 0.0),
        "receita_bandeira_maior_que_energia": F.col("receita_bandeira") > F.col("receita_energia"),
        "dias_atraso_negativo": F.col("dias_em_atraso") < 0,
    })

    return df, {"linhas": total, "data_corte": str(data_corte)}


executar_gold("gold_fato_faturamento_mensal", construir_gold_fato_faturamento_mensal)

## 3. `gold_reconciliacao_faturamento`
Ponte do faturamento e da energia, do número bruto até o usado nos indicadores, por competência (`grão: competencia`, `DEC-023`).

**Checks obrigatórios do contrato:**
- `receita_faturada_total = receita_faturada_cadastro + receita_faturada_orfas` ($\le$ R\$ 0,01)
- `receita_faturada_cadastro = receita_energia_cadastro + tributos_cadastro` ($\le$ R\$ 0,01)
- `kwh_faturado_total = kwh_leitura_total + kwh_correcao_troca_medidor + kwh_correcao_sinal_invertido` ($\le$ 0,001 kWh)

In [ ]:
COLUNAS_RECONCILIACAO = [
    "competencia",
    "receita_faturada_total",
    "receita_faturada_orfas",
    "receita_faturada_cadastro",
    "tributos_cadastro",
    "receita_energia_cadastro",
    "n_faturas_orfas",
    "kwh_leitura_total",
    "kwh_correcao_troca_medidor",
    "kwh_correcao_sinal_invertido",
    "kwh_faturado_total",
]


def construir_gold_reconciliacao_faturamento():
    faturas_todas = obter_silver("faturas")
    fato = obter_gold("gold_fato_faturamento_mensal")

    orfas_por_mes = (
        faturas_todas
        .groupBy(F.col("mes_referencia").alias("competencia"))
        .agg(
            money(F.sum("valor_total")).alias("receita_faturada_total"),
            money(F.sum(F.when(F.col("flag_uc_orfa"), F.col("valor_total")).otherwise(F.lit(0)))).alias("receita_faturada_orfas"),
            F.sum(F.when(F.col("flag_uc_orfa"), F.lit(1)).otherwise(F.lit(0))).cast("int").alias("n_faturas_orfas"),
        )
    )

    cadastro_por_mes = (
        fato
        .groupBy("competencia")
        .agg(
            money(F.sum("receita_faturada")).alias("receita_faturada_cadastro"),
            money(F.sum("tributos")).alias("tributos_cadastro"),
            money(F.sum("receita_energia")).alias("receita_energia_cadastro"),
            energy(F.sum("kwh_leitura")).alias("kwh_leitura_total"),
            energy(
                F.sum(
                    F.when(
                        F.col("origem_kwh") == "leitura_atual_troca_medidor",
                        F.col("kwh_faturado") - F.col("kwh_leitura"),
                    ).otherwise(F.lit(0))
                )
            ).alias("kwh_correcao_troca_medidor"),
            energy(
                F.sum(
                    F.when(
                        F.col("origem_kwh") == "consumo_absoluto",
                        F.col("kwh_faturado") - F.col("kwh_leitura"),
                    ).otherwise(F.lit(0))
                )
            ).alias("kwh_correcao_sinal_invertido"),
            energy(F.sum("kwh_faturado")).alias("kwh_faturado_total"),
        )
    )

    df = (
        orfas_por_mes
        .join(cadastro_por_mes, "competencia", "inner")
        .select(*COLUNAS_RECONCILIACAO)
        .orderBy("competencia")
    )

    exigir_colunas_exatas(df, COLUNAS_RECONCILIACAO)
    total = exigir_chave_unica(df, ["competencia"])
    exigir_sem_nulos(df, COLUNAS_RECONCILIACAO)

    exigir_zero(df, {
        "fechamento_receita_total": F.abs(F.col("receita_faturada_total") - F.col("receita_faturada_cadastro") - F.col("receita_faturada_orfas")) > 0.01,
        "fechamento_cadastro": F.abs(F.col("receita_faturada_cadastro") - F.col("receita_energia_cadastro") - F.col("tributos_cadastro")) > 0.01,
        "fechamento_kwh_total": F.abs(F.col("kwh_faturado_total") - F.col("kwh_leitura_total") - F.col("kwh_correcao_troca_medidor") - F.col("kwh_correcao_sinal_invertido")) > 0.001,
    })

    return df, {"linhas": total}


executar_gold("gold_reconciliacao_faturamento", construir_gold_reconciliacao_faturamento)

## 4. `gold_continuidade_alimentador_mensal`
DEC e FEC por alimentador e competência (`grão: cod_alimentador × competencia`).

**Regras regulatórias aplicadas (`DEC-014`, `DEC-027`, `DEC-030`):**
- `n_uc`: quantidade de **UCs faturadas** atendidas pelo alimentador na competência (denominador oficial do DEC/FEC segundo o PRODIST Módulo 8). Ficam excluídas as UCs da UC-12 (sem fatura/medidor) e meses antes da primeira fatura.
- `horas_interrompidas` e `n_interrupcoes`: agregadas a partir da base que exclui ocorrências registradas antes da ligação da UC (`OCO-04`).
- `dec = horas_interrompidas / n_uc`; `fec = n_interrupcoes / n_uc`.

In [ ]:
COLUNAS_CONTINUIDADE = [
    "cod_alimentador",
    "competencia",
    "dec",
    "fec",
    "n_uc",
    "n_interrupcoes",
    "horas_interrompidas",
]


def construir_gold_continuidade_alimentador_mensal():
    base = obter_base_uc_mes()

    df = (
        base
        .groupBy("cod_alimentador", "competencia")
        .agg(
            F.sum("n_faturas").cast("int").alias("n_uc"),
            F.sum("n_interrupcoes").cast("int").alias("n_interrupcoes"),
            ratio(F.sum("horas_interrompidas")).alias("horas_interrompidas"),
        )
        .withColumn("dec", ratio(F.col("horas_interrompidas") / F.col("n_uc")))
        .withColumn("fec", ratio(F.col("n_interrupcoes") / F.col("n_uc")))
        .select(*COLUNAS_CONTINUIDADE)
        .orderBy("cod_alimentador", "competencia")
    )

    exigir_colunas_exatas(df, COLUNAS_CONTINUIDADE)
    total = exigir_chave_unica(df, ["cod_alimentador", "competencia"])
    exigir_sem_nulos(df, COLUNAS_CONTINUIDADE)
    exigir_zero(df, {
        "n_uc_nao_positivo": F.col("n_uc") <= 0,
        "dec_negativo": F.col("dec") < 0,
        "fec_negativo": F.col("fec") < 0,
    })

    return df, {"linhas": total, "alimentadores": df.select("cod_alimentador").distinct().count()}


executar_gold("gold_continuidade_alimentador_mensal", construir_gold_continuidade_alimentador_mensal)

## 5. `gold_kpis_segmento_mensal`
KPIs por recorte multidimensional e competência, guardados exclusivamente como **componentes somáveis** (`DEC-023`, `DEC-027`, `DEC-030`).

**Grão:** `competencia × municipio × bairro × classe_cadastro × possui_gd × cod_alimentador × tipo_medidor`

**Regras aplicadas (`DEC-027`, `DEC-030`):**
- Chave primária expandida para 7 dimensões, permitindo cruzamento de continuidade, perdas e cadastro.
- `n_uc`: contagem de UCs faturadas no recorte (`F.sum("n_faturas")`).
- `receita_arrecadada`: nula quando o recorte não tem faturas na competência; `0.00` quando há fatura e nada foi pago (`DEC-030`).
- `soma_notas`: nula quando nenhum chamado do recorte foi avaliado (`n_notas == 0`) (`DEC-030`).
- Total de 26 colunas somáveis cobrindo energia, receita, tributos, bandeira, inadimplência e continuidade.

In [ ]:
COLUNAS_KPIS_SEGMENTO = [
    "competencia",
    "municipio",
    "bairro",
    "classe_cadastro",
    "classe",
    "possui_gd",
    "cod_alimentador",
    "tipo_medidor",
    "n_uc",
    "kwh_faturado",
    "receita_faturada",
    "receita_energia",
    "tributos",
    "receita_arrecadada",
    "horas_interrompidas",
    "n_interrupcoes",
    "n_chamados",
    "soma_notas",
    "n_notas",
    "receita_bandeira",
    "n_faturas",
    "n_faturas_nao_pagas",
    "valor_nao_pago",
    "n_faturas_pagas_com_atraso",
    "soma_dias_em_atraso",
    "n_leituras",
    "n_leituras_estimadas",
]

CHAVE_KPIS_SEGMENTO = [
    "competencia",
    "municipio",
    "bairro",
    "classe_cadastro",
    "possui_gd",
    "cod_alimentador",
    "tipo_medidor",
]


def construir_gold_kpis_segmento_mensal():
    base = obter_base_uc_mes()

    df = (
        base
        .groupBy(*CHAVE_KPIS_SEGMENTO)
        .agg(
            F.first("classe").alias("classe"),
            F.countDistinct("classe").alias("_classes"),
            F.sum("n_faturas").cast("int").alias("n_uc"),
            energy(F.sum("kwh_faturado")).alias("kwh_faturado"),
            money(F.sum("receita_faturada")).alias("receita_faturada"),
            money(F.sum("receita_energia")).alias("receita_energia"),
            money(F.sum("tributos")).alias("tributos"),
            money(F.sum("receita_arrecadada")).alias("_rec_arr"),
            ratio(F.sum("horas_interrompidas")).alias("horas_interrompidas"),
            F.sum("n_interrupcoes").cast("int").alias("n_interrupcoes"),
            F.sum("n_chamados").cast("int").alias("n_chamados"),
            ratio(F.sum("soma_notas")).alias("_soma_notas"),
            F.sum("n_notas").cast("int").alias("n_notas"),
            money(F.sum("receita_bandeira")).alias("receita_bandeira"),
            F.sum("n_faturas").cast("int").alias("n_faturas"),
            F.sum("n_faturas_nao_pagas").cast("int").alias("n_faturas_nao_pagas"),
            money(F.sum("valor_nao_pago")).alias("valor_nao_pago"),
            F.sum("n_faturas_pagas_com_atraso").cast("int").alias("n_faturas_pagas_com_atraso"),
            F.sum("soma_dias_em_atraso").cast("int").alias("soma_dias_em_atraso"),
            F.sum("n_leituras").cast("int").alias("n_leituras"),
            F.sum("n_leituras_estimadas").cast("int").alias("n_leituras_estimadas"),
        )
        .withColumn(
            "receita_arrecadada",
            F.when(F.col("n_faturas") > 0, F.col("_rec_arr")).otherwise(F.lit(None).cast("decimal(14,2)")),
        )
        .withColumn(
            "soma_notas",
            F.when(F.col("n_notas") > 0, F.col("_soma_notas")).otherwise(F.lit(None).cast("double")),
        )
    )

    exigir_zero(df, {"classe_cadastro_multiplas_classes": F.col("_classes") > 1})
    df = df.select(*COLUNAS_KPIS_SEGMENTO).orderBy(*CHAVE_KPIS_SEGMENTO)

    exigir_colunas_exatas(df, COLUNAS_KPIS_SEGMENTO)
    total = exigir_chave_unica(df, CHAVE_KPIS_SEGMENTO)
    exigir_sem_nulos(df, [c for c in COLUNAS_KPIS_SEGMENTO if c not in ["receita_arrecadada", "soma_notas"]])

    exigir_zero(df, {
        "fechamento_receita_segmento": F.abs(F.col("receita_faturada") - F.col("receita_energia") - F.col("tributos")) > 0.01,
        "n_leituras_menor_que_estimadas": F.col("n_leituras") < F.col("n_leituras_estimadas"),
        "n_faturas_menor_que_nao_pagas": F.col("n_faturas") < F.col("n_faturas_nao_pagas"),
    })

    return df, {"linhas": total}


executar_gold("gold_kpis_segmento_mensal", construir_gold_kpis_segmento_mensal)

## 6. `gold_kpis_mensais`
KPIs agregados no nível da empresa por competência (`grão: competencia`).

Calculado a partir de `gold_kpis_segmento_mensal`, garantindo total consistência com os filtros do portal e do agente.

In [ ]:
COLUNAS_KPIS_MENSAIS = [
    "competencia",
    "kwh_faturado",
    "receita_faturada",
    "receita_energia",
    "dec",
    "fec",
    "n_chamados",
    "satisfacao_media",
]


def construir_gold_kpis_mensais():
    seg = obter_gold("gold_kpis_segmento_mensal")
    cobertura_atd = (
        obter_silver("atendimentos")
        .agg(
            F.date_format(F.min("data_abertura"), "yyyy-MM").alias("ini"),
            F.date_format(F.max("data_abertura"), "yyyy-MM").alias("fim"),
        )
        .first()
    )
    tem_cobertura_atd = (
        F.col("competencia").between(cobertura_atd["ini"], cobertura_atd["fim"])
        if cobertura_atd and cobertura_atd["ini"] is not None
        else F.lit(False)
    )

    df = (
        seg
        .groupBy("competencia")
        .agg(
            energy(F.sum("kwh_faturado")).alias("kwh_faturado"),
            money(F.sum("receita_faturada")).alias("receita_faturada"),
            money(F.sum("receita_energia")).alias("receita_energia"),
            ratio(F.sum("horas_interrompidas") / F.sum("n_uc")).alias("dec"),
            ratio(F.sum("n_interrupcoes") / F.sum("n_uc")).alias("fec"),
            F.sum("n_chamados").cast("int").alias("_chamados"),
            F.sum("soma_notas").alias("_soma_notas"),
            F.sum("n_notas").alias("_n_notas"),
        )
        .withColumn(
            "n_chamados",
            F.when(tem_cobertura_atd, F.col("_chamados")).otherwise(F.lit(None).cast("int")),
        )
        .withColumn(
            "satisfacao_media",
            ratio(
                F.when(tem_cobertura_atd & (F.col("_n_notas") > 0), F.col("_soma_notas") / F.col("_n_notas"))
                 .otherwise(F.lit(None).cast("double"))
            ),
        )
        .select(*COLUNAS_KPIS_MENSAIS)
        .orderBy("competencia")
    )

    exigir_colunas_exatas(df, COLUNAS_KPIS_MENSAIS)
    total = exigir_chave_unica(df, ["competencia"])
    exigir_sem_nulos(df, ["competencia", "kwh_faturado", "receita_faturada", "receita_energia"])

    janela_atd = f"{cobertura_atd['ini']} a {cobertura_atd['fim']}" if cobertura_atd and cobertura_atd["ini"] else "sem atendimentos"
    return df, {"linhas": total, "cobertura_atendimentos": janela_atd}


executar_gold("gold_kpis_mensais", construir_gold_kpis_mensais)

## 7. `gold_receita_decomposicao`
Decomposição PVM em lote da variação de receita de energia entre períodos (`DEC-014`, `DEC-023`, `DEC-028`, `DEC-029`, `DEC-031`).

**Grão:** `periodo_base × periodo_atual × municipio × bairro × classe × possui_gd`

**Regras aplicadas:**
- **Gravação em lote (`DEC-028` / `DEC-031`):** Grava todos os pares de "mesmos meses em anos diferentes" com duração de 1 a 12 meses.
- **`possui_gd` fixo nos dois períodos (`DEC-023`):** É o estado no último mês do período atual (`fim_atual`), comparando a mesma população nos dois lados para que o efeito GD seja a perda de volume das UCs com GD.
- **Efeito bandeira e efeito tarifa (`DEC-029`):** A bandeira é isolada como acréscimo fixo por kWh. `efeito_preco = efeito_tarifa + efeito_bandeira`.
- **Identidade algébrica exata:** $\text{efeito\_preco} + \text{efeito\_volume} + \text{efeito\_mix} \equiv \text{receita\_atual} - \text{receita\_base}$.

In [ ]:
COLUNAS_DECOMPOSICAO = [
    "periodo_base",
    "periodo_atual",
    "municipio",
    "bairro",
    "classe",
    "possui_gd",
    "receita_base",
    "receita_atual",
    "efeito_preco",
    "efeito_tarifa",
    "efeito_bandeira",
    "efeito_volume",
    "efeito_mix",
]


def gerar_pares_mesmos_meses(comps):
    """Gera todos os pares de mesmos meses em anos diferentes com durações de 1 a 12 meses (DEC-028)."""
    linhas, n = [], 0
    for duracao in range(1, 13):
        for i in range(len(comps)):
            for anos in range(1, 5):
                j = i + 12 * anos
                if j + duracao > len(comps) or i + duracao > len(comps):
                    continue
                base, atual = comps[i:i + duracao], comps[j:j + duracao]
                pb, pa = f"{base[0]}/{base[-1]}", f"{atual[0]}/{atual[-1]}"
                linhas += [(pb, pa, c, "0", atual[-1]) for c in base] + [(pb, pa, c, "1", atual[-1]) for c in atual]
                n += 1
    df_pares = spark.createDataFrame(
        linhas,
        "periodo_base string, periodo_atual string, competencia string, lado string, fim_atual string"
    )
    return df_pares, n


def construir_gold_receita_decomposicao():
    fato = obter_gold("gold_fato_faturamento_mensal")
    dim_uc = obter_gold("gold_dim_uc")
    gd = obter_silver("geracao_distribuida").select(
        F.col("id_uc").alias("cod_uc"),
        F.date_format("data_conexao", "yyyy-MM").alias("_mes_conexao"),
    )

    comps = sorted(r[0] for r in fato.select("competencia").distinct().collect())
    pares_df, n_pares = gerar_pares_mesmos_meses(comps)
    assert n_pares > 0, f"nenhum par de mesmos meses em {len(comps)} competências"

    seg = ["municipio", "bairro", "classe"]
    mensal = (
        fato
        .join(dim_uc.select("cod_uc", *seg), "cod_uc")
        .join(gd, "cod_uc", "left")
        .groupBy(*seg, "_mes_conexao", "competencia")
        .agg(
            F.sum(F.col("kwh_faturado").cast("double")).alias("q"),
            F.sum("receita_energia").alias("receita"),
            F.sum(F.col("receita_energia").cast("double")).alias("r"),
            F.sum(F.col("receita_bandeira").cast("double")).alias("b"),
        )
    )

    par = ["periodo_base", "periodo_atual"]
    p = (
        mensal
        .join(F.broadcast(pares_df), "competencia")
        .withColumn("possui_gd", F.coalesce(F.col("_mes_conexao") <= F.col("fim_atual"), F.lit(False)))
        .groupBy(*par, *seg, "possui_gd")
        .pivot("lado", ["0", "1"])
        .agg(
            F.sum("q").alias("q"),
            F.sum("r").alias("r"),
            F.sum("b").alias("b"),
            F.sum("receita").alias("receita"),
        )
        .fillna(0)
    )

    todas = Window.partitionBy(*par)
    classe_win = Window.partitionBy(*par, "classe")
    q0c, q1c = F.sum("0_q").over(classe_win), F.sum("1_q").over(classe_win)
    q0t, q1t = F.sum("0_q").over(todas), F.sum("1_q").over(todas)

    tem_base = F.col("0_q") > 0
    rb0, rb1 = F.col("0_r") - F.col("0_b"), F.col("1_r") - F.col("1_b")
    p0 = F.when(tem_base, F.col("0_r") / F.col("0_q")).otherwise(F.sum("0_r").over(classe_win) / q0c)
    pb0 = F.when(tem_base, rb0 / F.col("0_q")).otherwise(F.sum(rb0).over(classe_win) / q0c)

    p = (
        p
        .withColumn("_p0", p0)
        .withColumn("_pb0", pb0)
        .withColumn("_b0", F.col("_p0") - F.col("_pb0"))
        .withColumn("_mix", q1t * (q1c / q1t - q0c / q0t) * (F.col("0_q") / q0c) * F.col("_p0"))
    )

    df = (
        p
        .select(
            *par,
            *seg,
            "possui_gd",
            money(F.col("0_receita")).alias("receita_base"),
            money(F.col("1_receita")).alias("receita_atual"),
            money(F.round((rb1 - F.col("1_q") * F.col("_pb0")) - (rb0 - F.col("0_q") * F.col("_pb0")), 2)).alias("efeito_tarifa"),
            money(F.round((F.col("1_b") - F.col("1_q") * F.col("_b0")) - (F.col("0_b") - F.col("0_q") * F.col("_b0")), 2)).alias("efeito_bandeira"),
            money(F.round((F.col("1_q") - F.col("0_q")) * F.col("_p0") - F.col("_mix"), 2)).alias("efeito_volume"),
            money(F.round(F.col("_mix"), 2)).alias("efeito_mix"),
        )
        .withColumn("efeito_preco", money(F.col("efeito_tarifa") + F.col("efeito_bandeira")))
        .select(*COLUNAS_DECOMPOSICAO)
        .orderBy(*par, *seg, "possui_gd")
    )

    exigir_colunas_exatas(df, COLUNAS_DECOMPOSICAO)
    chave = [*par, *seg, "possui_gd"]
    total = exigir_chave_unica(df, chave)
    exigir_sem_nulos(df, COLUNAS_DECOMPOSICAO)

    exigir_zero(df, {
        "preco_diferente_de_tarifa_mais_bandeira": F.abs(F.col("efeito_preco") - F.col("efeito_tarifa") - F.col("efeito_bandeira")) > 0.01,
        "fechamento_efeitos_pvm": F.abs(F.col("efeito_preco") + F.col("efeito_volume") + F.col("efeito_mix") - (F.col("receita_atual") - F.col("receita_base"))) > 0.02,
    })

    return df, {"linhas": total, "pares_calculados": n_pares}


executar_gold("gold_receita_decomposicao", construir_gold_receita_decomposicao)

## 8. `gold_fila_suspeitas`
UCs suspeitas de perda não técnica ordenadas por prioridade (`score_suspeita = 1..N`) (`DEC-030`).

**Regras calibradas no histórico de campo (`DEC-030`):**
- **Quem entra:** UCs sem GD, com medidor ativo, com pelo menos 6 meses de histórico em cada janela e queda $\ge 15\%$ no consumo médio (`kwh_faturado`) das 12 últimas competências contra as 12 anteriores.
- **Quem sai:** UCs com irregularidade confirmada (já sanadas) e UCs inspecionadas sem irregularidade nos 12 meses antes do fim dos dados.
- **Pontuação e Ordenação:**
  - Queda $\ge 30\%$: 3 pontos; queda entre 15% e 30%: 1 ponto;
  - Medidor eletromecânico: 1 ponto; $\ge 25\%$ de leituras estimadas: 1 ponto;
  - Ordenação por pontos de risco (desc), maior `kwh_recuperavel_est` (desc) e `cod_uc` (asc).

In [ ]:
COLUNAS_FILA_SUSPEITAS = [
    "cod_uc",
    "score_suspeita",
    "motivos",
    "kwh_recuperavel_est",
]

QUEDA_ENTRADA = 0.15
QUEDA_FORTE = 0.30
ESTIMADAS_ALTA = 0.25
MESES_MINIMOS = 6


def validar_na_data_da_inspecao(fato, inspecao):
    """Precisão e cobertura da regra de entrada com a janela que a equipe via: 12 competências antes da inspeção x 12 anteriores."""
    gd = obter_gold("gold_dim_uc").filter("possui_gd").select("cod_uc")
    distancia = F.months_between(
        F.to_date(F.concat(F.col("_mes_inspecao"), F.lit("-01"))),
        F.to_date(F.concat(F.col("competencia"), F.lit("-01"))),
    ).cast("int")
    janela = (
        fato.select("cod_uc", "competencia", "kwh_faturado")
        .join(inspecao, "cod_uc")
        .join(gd, "cod_uc", "left_anti")
        .withColumn("_d", distancia)
        .groupBy("cod_uc", "_resultado")
        .agg(
            F.avg(F.when(F.col("_d").between(1, 12), F.col("kwh_faturado"))).alias("_recente"),
            F.avg(F.when(F.col("_d").between(13, 24), F.col("kwh_faturado"))).alias("_anterior"),
            F.count(F.when(F.col("_d").between(1, 12), 1)).alias("_n_recente"),
            F.count(F.when(F.col("_d").between(13, 24), 1)).alias("_n_anterior"),
        )
        .filter((F.col("_n_recente") >= MESES_MINIMOS) & (F.col("_n_anterior") >= MESES_MINIMOS) & (F.col("_anterior") > 0))
    )
    candidato = (1 - F.col("_recente") / F.col("_anterior")) >= QUEDA_ENTRADA
    confirmada = F.col("_resultado") == "IRREGULARIDADE CONFIRMADA"
    v = janela.agg(
        F.sum((candidato & confirmada).cast("int")).alias("acertos"),
        F.sum(candidato.cast("int")).alias("apontadas"),
        F.sum(confirmada.cast("int")).alias("confirmadas"),
        F.count("*").alias("inspecionadas_medidas"),
    ).first()
    return {
        "precisao_na_data_da_inspecao": round(v["acertos"] / v["apontadas"], 3) if v["apontadas"] else None,
        "cobertura_na_data_da_inspecao": round(v["acertos"] / v["confirmadas"], 3) if v["confirmadas"] else None,
        "inspecionadas_medidas": v["inspecionadas_medidas"],
    }


def construir_gold_fila_suspeitas():
    fato = obter_gold("gold_fato_faturamento_mensal")
    comps = sorted(r[0] for r in fato.select("competencia").distinct().collect())
    recentes, anteriores = comps[-12:], comps[-24:-12]

    consumo = fato.groupBy("cod_uc").agg(
        F.avg(F.when(F.col("competencia").isin(recentes), F.col("kwh_faturado"))).alias("_recente"),
        F.avg(F.when(F.col("competencia").isin(anteriores), F.col("kwh_faturado"))).alias("_anterior"),
        F.count(F.when(F.col("competencia").isin(recentes), 1)).alias("_n_recente"),
        F.count(F.when(F.col("competencia").isin(anteriores), 1)).alias("_n_anterior"),
    )

    estimadas = (
        obter_silver("leituras")
        .filter(F.col("mes_referencia").isin(recentes))
        .groupBy(F.col("id_uc").alias("cod_uc"))
        .agg(F.avg((F.col("tipo_leitura") == "ESTIMADA").cast("double")).alias("_estimadas"))
    )

    medidor = (
        obter_silver("medidores")
        .filter(F.col("status") == "ATIVO")
        .select(
            F.col("id_uc").alias("cod_uc"),
            (F.col("tipo") == "ELETROMECANICO").alias("_eletromecanico"),
        )
    )

    inspecao = (
        obter_silver("inspecoes")
        .select(
            F.col("id_uc").alias("cod_uc"),
            F.col("resultado").alias("_resultado"),
            F.date_format("data", "yyyy-MM").alias("_mes_inspecao"),
        )
    )

    base = (
        obter_gold("gold_dim_uc")
        .filter(~F.col("possui_gd"))
        .select("cod_uc")
        .join(consumo, "cod_uc")
        .join(estimadas, "cod_uc", "left")
        .join(medidor, "cod_uc")
        .join(inspecao, "cod_uc", "left")
        .filter((F.col("_n_recente") >= MESES_MINIMOS) & (F.col("_n_anterior") >= MESES_MINIMOS) & (F.col("_anterior") > 0))
        .withColumn("_queda", (1 - F.col("_recente") / F.col("_anterior")).cast("double"))
    )
    candidato = F.col("_queda") >= QUEDA_ENTRADA

    validacao = validar_na_data_da_inspecao(fato, inspecao)

    forte = F.col("_queda") >= QUEDA_FORTE
    eletromecanico = F.coalesce(F.col("_eletromecanico"), F.lit(False))
    muitas_estimadas = F.coalesce(F.col("_estimadas") >= ESTIMADAS_ALTA, F.lit(False))
    pontos = (F.when(forte, 3).otherwise(1) + eletromecanico.cast("int") + muitas_estimadas.cast("int"))

    motivos = F.filter(
        F.array(
            F.format_string("Queda de %.0f%% no consumo médio das 12 últimas competências contra as 12 anteriores", F.col("_queda") * 100),
            F.when(eletromecanico, F.lit("Medidor eletromecânico")),
            F.when(muitas_estimadas, F.format_string("%.0f%% das leituras das 12 últimas competências foram estimadas", F.col("_estimadas") * 100)),
        ),
        lambda m: m.isNotNull(),
    )
    recuperavel_bruto = F.greatest(F.col("_anterior") - F.col("_recente"), F.lit(0))

    confirmada = F.col("_resultado") == "IRREGULARIDADE CONFIRMADA"
    recente = F.col("_mes_inspecao") >= F.lit(recentes[0])
    fora_da_fila = F.coalesce(confirmada | recente, F.lit(False))

    df = (
        base
        .filter(candidato & ~fora_da_fila)
        .withColumn("_pontos", pontos)
        .withColumn("_recuperavel_bruto", recuperavel_bruto)
        .withColumn("kwh_recuperavel_est", energy(F.col("_recuperavel_bruto")))
        .withColumn("motivos", motivos)
        .withColumn(
            "score_suspeita",
            F.row_number().over(
                Window.orderBy(F.desc("_pontos"), F.desc("_recuperavel_bruto"), "cod_uc")
            ).cast("int"),
        )
        .select(*COLUNAS_FILA_SUSPEITAS)
        .orderBy("score_suspeita")
    )

    exigir_colunas_exatas(df, COLUNAS_FILA_SUSPEITAS)
    total = exigir_chave_unica(df, ["cod_uc"])
    exigir_sem_nulos(df, ["cod_uc", "score_suspeita", "motivos"])
    faixa = df.agg(
        F.min("score_suspeita").alias("min"),
        F.max("score_suspeita").alias("max"),
        F.countDistinct("score_suspeita").alias("n"),
    ).first()
    assert total == 0 or (faixa["min"] == 1 and faixa["max"] == total == faixa["n"]), f"score_suspeita fora de 1..N: {faixa}"
    exigir_zero(df, {"sem_motivo": F.size("motivos") == 0, "recuperavel_negativo": F.col("kwh_recuperavel_est") < 0})
    exigir_fk_uc(df, "gold_fila_suspeitas")

    return df, {
        "linhas": total,
        "janela": f"{recentes[0]}/{recentes[-1]} contra {anteriores[0]}/{anteriores[-1]}",
        **validacao,
    }


executar_gold("gold_fila_suspeitas", construir_gold_fila_suspeitas)


## 9. Tabelas de Detalhe da UC 360 e Perdas (`DEC-027`, `DEC-030`)
Tabelas granulares previstas no Contrato da Gold v0.3.0 para visão individual da UC e gestão de inspeções:
- `gold_medidor_uc`: medidor vigente de cada UC com `n_trocas` e `data_ultima_troca`;
- `gold_interrupcoes_uc`: histórico de interrupções da UC (excluindo eventos antes da ligação, `OCO-04`);
- `gold_atendimentos_uc`: histórico de chamados (sem campos de texto livre sensíveis);
- `gold_inspecoes_uc`: histórico de inspeções de campo por UC.

In [ ]:
COLUNAS_MEDIDOR_UC = [
    "cod_uc",
    "cod_medidor",
    "tipo_medidor",
    "modelo",
    "data_instalacao",
    "idade_anos",
    "n_trocas",
    "data_ultima_troca",
]

COLUNAS_INTERRUPCOES_UC = [
    "cod_ocorrencia",
    "cod_uc",
    "cod_alimentador",
    "competencia",
    "data_abertura",
    "data_encerramento",
    "duracao_horas",
    "tipo",
    "causa",
]

COLUNAS_ATENDIMENTOS_UC = [
    "protocolo",
    "cod_uc",
    "competencia",
    "data_abertura",
    "data_fechamento",
    "canal",
    "assunto",
    "subassunto",
    "nota_satisfacao",
]

COLUNAS_INSPECOES_UC = [
    "cod_inspecao",
    "cod_uc",
    "data_inspecao",
    "competencia",
    "resultado",
    "tipo_irregularidade",
    "kwh_recuperado",
]

RESULTADO_INSP_MAP = {
    "IRREGULARIDADE CONFIRMADA": "irregularidade_confirmada",
    "SEM IRREGULARIDADE": "sem_irregularidade",
}


def construir_gold_medidor_uc():
    fato = obter_gold("gold_fato_faturamento_mensal")
    ultima_comp = fato.agg(F.max("competencia")).first()[0]
    data_ref = F.last_day(F.to_date(F.lit(f"{ultima_comp}-01")))

    m = obter_silver("medidores")
    trocas = (
        m
        .filter(F.col("status") == "RETIRADO")
        .groupBy(F.col("id_uc").alias("cod_uc"))
        .agg(
            F.count(F.lit(1)).cast("int").alias("n_trocas"),
            F.max(F.col("data_retirada").cast("date")).alias("data_ultima_troca"),
        )
    )

    df = (
        m
        .filter(F.col("status") == "ATIVO")
        .select(
            F.col("id_uc").alias("cod_uc"),
            F.col("id_medidor").alias("cod_medidor"),
            F.lower(F.col("tipo")).alias("tipo_medidor"),
            F.col("modelo"),
            F.col("data_instalacao").cast("date").alias("data_instalacao"),
            ratio(F.months_between(data_ref, F.col("data_instalacao")) / 12.0).alias("idade_anos"),
        )
        .join(trocas, "cod_uc", "left")
        .fillna(0, ["n_trocas"])
        .select(*COLUNAS_MEDIDOR_UC)
        .orderBy("cod_uc")
    )

    exigir_colunas_exatas(df, COLUNAS_MEDIDOR_UC)
    total = exigir_chave_unica(df, ["cod_uc"])
    exigir_sem_nulos(df, ["cod_uc", "cod_medidor", "tipo_medidor", "data_instalacao", "idade_anos", "n_trocas"])
    exigir_fk_uc(df, "gold_medidor_uc")
    exigir_zero(df, {"idade_negativa": F.col("idade_anos") < 0})

    return df, {"linhas": total}


def construir_gold_interrupcoes_uc():
    dim_uc = obter_gold("gold_dim_uc").select(
        "cod_uc",
        "cod_alimentador",
        F.date_format("data_ligacao", "yyyy-MM").alias("_mes_ligacao"),
    )
    oc = obter_silver("ocorrencias_rede")

    df = (
        oc
        .select(
            F.col("id_ocorrencia").alias("cod_ocorrencia"),
            F.col("id_uc").alias("cod_uc"),
            F.date_format(F.col("data_abertura"), "yyyy-MM").alias("competencia"),
            F.col("data_abertura").cast("timestamp").alias("data_abertura"),
            F.col("data_encerramento").cast("timestamp").alias("data_encerramento"),
            ratio(F.col("duracao_min") / 60.0).alias("duracao_horas"),
            F.col("tipo"),
            F.col("causa"),
        )
        .join(dim_uc, "cod_uc", "inner")
        .filter(F.col("competencia") >= F.col("_mes_ligacao"))
        .select(*COLUNAS_INTERRUPCOES_UC)
        .orderBy("cod_uc", "data_abertura", "cod_ocorrencia")
    )

    exigir_colunas_exatas(df, COLUNAS_INTERRUPCOES_UC)
    total = exigir_chave_unica(df, ["cod_ocorrencia"])
    exigir_sem_nulos(df, ["cod_ocorrencia", "cod_uc", "cod_alimentador", "competencia", "data_abertura", "data_encerramento", "duracao_horas", "tipo"])
    exigir_fk_uc(df, "gold_interrupcoes_uc")

    total_silver = oc.count()
    excluidas_oco04 = total_silver - total
    if "ocorrencias_antes_ligacao" in CACHE_AUX:
        assert excluidas_oco04 == CACHE_AUX["ocorrencias_antes_ligacao"], (
            f"OCO-04 inconsistente: excluídas em interrupcoes_uc ({excluidas_oco04:,}) "
            f"!= antes da ligação na base ({CACHE_AUX['ocorrencias_antes_ligacao']:,})"
        )

    return df, {"linhas": total, "excluidas_antes_da_ligacao": excluidas_oco04}


def construir_gold_atendimentos_uc():
    at = obter_silver("atendimentos")

    df = (
        at
        .select(
            F.col("protocolo"),
            F.col("id_uc").alias("cod_uc"),
            F.date_format(F.col("data_abertura"), "yyyy-MM").alias("competencia"),
            F.col("data_abertura").cast("timestamp").alias("data_abertura"),
            F.col("data_fechamento").cast("timestamp").alias("data_fechamento"),
            F.col("canal"),
            F.col("assunto"),
            F.col("subassunto"),
            ratio("nota_satisfacao").alias("nota_satisfacao"),
        )
        .select(*COLUNAS_ATENDIMENTOS_UC)
        .orderBy("cod_uc", "data_abertura", "protocolo")
    )

    exigir_colunas_exatas(df, COLUNAS_ATENDIMENTOS_UC)
    total = exigir_chave_unica(df, ["protocolo"])
    exigir_sem_nulos(df, ["protocolo", "cod_uc", "competencia", "data_abertura", "data_fechamento", "canal", "assunto"])
    exigir_fk_uc(df, "gold_atendimentos_uc")

    return df, {"linhas": total}


def construir_gold_inspecoes_uc():
    insp = obter_silver("inspecoes")
    mapa_resultado_expr = F.create_map(*[F.lit(x) for kv in RESULTADO_INSP_MAP.items() for x in kv])

    df = (
        insp
        .select(
            F.col("id_inspecao").alias("cod_inspecao"),
            F.col("id_uc").alias("cod_uc"),
            F.col("data").cast("date").alias("data_inspecao"),
            F.date_format(F.col("data"), "yyyy-MM").alias("competencia"),
            mapa_resultado_expr[F.col("resultado")].alias("resultado"),
            F.col("tipo_irregularidade"),
            energy(F.coalesce(F.col("kwh_recuperado"), F.lit(0))).alias("kwh_recuperado"),
        )
        .select(*COLUNAS_INSPECOES_UC)
        .orderBy("cod_uc", "data_inspecao", "cod_inspecao")
    )

    exigir_colunas_exatas(df, COLUNAS_INSPECOES_UC)
    total = exigir_chave_unica(df, ["cod_inspecao"])
    exigir_sem_nulos(df, ["cod_inspecao", "cod_uc", "data_inspecao", "competencia", "resultado", "kwh_recuperado"])
    exigir_fk_uc(df, "gold_inspecoes_uc")

    return df, {"linhas": total}


executar_gold("gold_medidor_uc", construir_gold_medidor_uc)
executar_gold("gold_interrupcoes_uc", construir_gold_interrupcoes_uc)
executar_gold("gold_atendimentos_uc", construir_gold_atendimentos_uc)
executar_gold("gold_inspecoes_uc", construir_gold_inspecoes_uc)

## 10. Validação Cruzada da Gold e Exportação CSV (`DEC-004`, `DEC-014`)
Conferência matemática entre todas as 12 tabelas da camada Gold:
1. `gold_fato_faturamento_mensal` × `gold_reconciliacao_faturamento` (soma mensal de kWh, receita faturada, receita de energia e tributos).
2. `gold_kpis_segmento_mensal` × `gold_kpis_mensais` (soma mensal de kWh, receita faturada e receita de energia).
3. `gold_kpis_segmento_mensal` × `gold_continuidade_alimentador_mensal` (soma mensal de `n_uc`, `horas_interrompidas` e `n_interrupcoes`).
4. `gold_receita_decomposicao` × `gold_fato_faturamento_mensal` (soma de `receita_base` e `receita_atual` em todos os pares calculados).
5. Se `exportar_csv = sim`, gera um arquivo `.csv` por tabela em `csv_export_path` (12 arquivos).

In [ ]:
def validar_cruzado_gold():
    print("=== validacao_cruzada_gold")
    try:
        fato = obter_gold("gold_fato_faturamento_mensal")
        rec = obter_gold("gold_reconciliacao_faturamento")
        seg = obter_gold("gold_kpis_segmento_mensal")
        kpis = obter_gold("gold_kpis_mensais")
        cont = obter_gold("gold_continuidade_alimentador_mensal")

        # 1. Fato x Reconciliação
        f_mes = fato.groupBy("competencia").agg(
            money(F.sum("receita_faturada")).alias("f_rec_fat"),
            money(F.sum("receita_energia")).alias("f_rec_ene"),
            money(F.sum("tributos")).alias("f_trib"),
            energy(F.sum("kwh_faturado")).alias("f_kwh"),
        )
        exigir_zero(rec.join(f_mes, "competencia", "full_outer"), {
            "rec_x_fato_receita_faturada": F.col("receita_faturada_cadastro") != F.col("f_rec_fat"),
            "rec_x_fato_receita_energia": F.col("receita_energia_cadastro") != F.col("f_rec_ene"),
            "rec_x_fato_tributos": F.col("tributos_cadastro") != F.col("f_trib"),
            "rec_x_fato_kwh": F.col("kwh_faturado_total") != F.col("f_kwh"),
        })

        # 2. Segmento x KPIs Mensais
        s_mes = seg.groupBy("competencia").agg(
            energy(F.sum("kwh_faturado")).alias("s_kwh"),
            money(F.sum("receita_faturada")).alias("s_rec_fat"),
            money(F.sum("receita_energia")).alias("s_rec_ene"),
            F.sum("n_uc").alias("s_n_uc"),
            F.sum("n_interrupcoes").alias("s_n_int"),
            ratio(F.sum("horas_interrompidas")).alias("s_horas"),
        )
        exigir_zero(kpis.join(s_mes, "competencia", "full_outer"), {
            "kpis_x_seg_kwh": F.col("kwh_faturado") != F.col("s_kwh"),
            "kpis_x_seg_receita_faturada": F.col("receita_faturada") != F.col("s_rec_fat"),
            "kpis_x_seg_receita_energia": F.col("receita_energia") != F.col("s_rec_ene"),
        })

        # 3. Segmento x Continuidade por Alimentador
        c_mes = cont.groupBy("competencia").agg(
            F.sum("n_uc").alias("c_n_uc"),
            F.sum("n_interrupcoes").alias("c_n_int"),
            ratio(F.sum("horas_interrompidas")).alias("c_horas"),
        )
        exigir_zero(s_mes.join(c_mes, "competencia", "full_outer"), {
            "seg_x_cont_n_uc": F.col("s_n_uc") != F.col("c_n_uc"),
            "seg_x_cont_n_interrupcoes": F.col("s_n_int") != F.col("c_n_int"),
            "seg_x_cont_horas": F.abs(F.col("s_horas") - F.col("c_horas")) > 1e-4,
        })

        # 4. Interrupções UC 360 x Continuidade por Alimentador (OCO-04)
        inter_uc = obter_gold("gold_interrupcoes_uc")
        i_alim = inter_uc.groupBy("competencia", "cod_alimentador").agg(
            F.count(F.lit(1)).alias("i_n"),
            ratio(F.sum("duracao_horas")).alias("i_horas"),
        )
        cont_ativas = cont.filter(F.col("n_interrupcoes") > 0).select(
            "competencia", "cod_alimentador",
            F.col("n_interrupcoes").alias("c_n"),
            F.col("horas_interrompidas").alias("c_h"),
        )
        exigir_zero(i_alim.join(cont_ativas, ["competencia", "cod_alimentador"], "full_outer"), {
            "inter_uc_x_cont_contagem": F.coalesce(F.col("i_n") != F.col("c_n"), F.lit(True)),
            "inter_uc_x_cont_horas": F.coalesce(F.abs(F.col("i_horas") - F.col("c_h")) > 1e-5, F.lit(True)),
        })

        # 5. Atendimentos UC 360 x Segmentos x Silver
        atd_uc = obter_gold("gold_atendimentos_uc")
        total_atd_uc = atd_uc.count()
        total_atd_seg = seg.agg(F.sum("n_chamados")).first()[0]
        assert total_atd_uc == total_atd_seg, (
            f"Total de chamados diverge entre gold_atendimentos_uc ({total_atd_uc:,}) e gold_kpis_segmento_mensal ({total_atd_seg:,})"
        )

        RESULTADOS["validacao_cruzada_gold"] = {"status": "conferida"}
        print("[OK] validacao_cruzada_gold: todas as tabelas reconciliadas com sucesso!")
    except Exception as e:
        RESULTADOS["validacao_cruzada_gold"] = {"status": "FALHA", "erro": f"{type(e).__name__}: {e}"}
        print("[FALHA] validacao_cruzada_gold")
        traceback.print_exc()


def exportar_csvs_contrato():
    if not EXPORTAR_CSV:
        print("Exportação CSV desativada (exportar_csv = nao).")
        return
    import os
    import shutil

    if CSV_EXPORT_PATH.startswith(f"/Volumes/{CATALOG}/default/"):
        vol_name = CSV_EXPORT_PATH.split("/")[4]
        spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.default.{vol_name}")
    os.makedirs(CSV_EXPORT_PATH, exist_ok=True)

    for nome in TABELAS_ESPERADAS:
        df = obter_gold(nome)
        cols_csv = []
        for field in df.schema.fields:
            dtype = field.dataType.simpleString()
            if dtype.startswith("array<"):
                cols_csv.append(F.concat_ws(";", F.col(field.name)).alias(field.name))
            elif dtype == "timestamp":
                # Formato sem fuso (DEC-004 / gold_schema.yaml: timestamp sem fuso)
                cols_csv.append(F.date_format(F.col(field.name), "yyyy-MM-dd'T'HH:mm:ss").alias(field.name))
            elif dtype == "date":
                cols_csv.append(F.date_format(F.col(field.name), "yyyy-MM-dd").alias(field.name))
            else:
                cols_csv.append(F.col(field.name))
        df_export = df.select(*cols_csv)

        tmp_dir = f"{CSV_EXPORT_PATH}/_tmp_{nome}"
        final_csv = f"{CSV_EXPORT_PATH}/{nome}.csv"
        (
            df_export.coalesce(1)
            .write.mode("overwrite")
            .option("header", "true")
            .option("delimiter", ",")
            .option("encoding", "UTF-8")
            .option("nullValue", "")
            .option("emptyValue", "")
            .csv(tmp_dir)
        )
        part_files = [f for f in os.listdir(tmp_dir) if f.startswith("part-") and f.endswith(".csv")]
        assert len(part_files) == 1, f"Esperado 1 part-file para {nome}, encontrado: {part_files}"
        shutil.move(f"{tmp_dir}/{part_files[0]}", final_csv)
        shutil.rmtree(tmp_dir, ignore_errors=True)
        print(f"[CSV] Exportado: {final_csv}")


validar_cruzado_gold()
exportar_csvs_contrato()

## 11. Resumo
Imprime o estado de cada tabela da camada Gold e da reconciliação cruzada. Output **confidencial** (`DEC-024`): limpe os outputs antes de commitar.
Falha explicitamente se alguma tabela não foi executada ou falhou.

In [ ]:
print("CONFIDENCIAL: resultados derivados do dataset. Não versionar outputs (DEC-024).")
print(f"schema_gold = {SCHEMA_GOLD} | gravar = {'sim' if GRAVAR else 'nao'} | {datetime.now().isoformat(timespec='seconds')}")
esperadas = TABELAS_ESPERADAS + ["validacao_cruzada_gold"]
incompletas = []
for nome in esperadas:
    r = RESULTADOS.get(nome)
    if r is None:
        print(f"- {nome}: NÃO EXECUTADA")
        incompletas.append(nome)
        continue
    print(f"- {nome}: {r}")
    if r["status"] == "FALHA":
        incompletas.append(nome)

assert not incompletas, f"Gold NÃO concluída. Não executadas ou com falha: {incompletas}"
print(f"[OK] Todas as 12 tabelas da Gold construídas e validadas" + (f" e gravadas em {GOLD}." if GRAVAR else " (nada foi gravado: gravar = nao)."))